# Honest Evaluation & Model Hardening

Implements Tier 1 + Tier 2 of the plan. Each section is independent; run top-to-bottom first, then you can rerun any section.

1. **Speaker-leakage scan** — detect cross-batch speaker overlap (biggest invisible-inflation risk)
2. **Load batches + base-model registry** — shared setup for everything below
3. **2-rotation × repeated 5×5-fold CV** with bootstrap threshold CIs — honest evaluation
4. **Per-batch score histograms** — visualise where batch shift lives
5. **Ensemble-of-seeds** on whisper_wp_xgb — cheap variance reduction
6. **PCA diagnostic** on 1024-d Whisper — is the full dim needed?
7. **WavLM variant comparison** — pre vs ft, whole vs seg
8. **Isotonic calibration** + reliability diagram
9. **Final winner recipe** with rotation spread

Outputs saved under `checkpoints_honest_eval/`.

In [ ]:
from pathlib import Path
import json, itertools, warnings, re, copy
import numpy as np
import pandas as pd
from collections import defaultdict

import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score, average_precision_score)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

warnings.filterwarnings('ignore')

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_honest_eval'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BATCHES = ['audios2', 'audios4', 'audios5']

# Training prior-adjustment kept consistent with fusion notebook
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE

# Evaluation knobs
N_SEEDS = 3     # repeated K-fold seeds (3 keeps runtime manageable; bump to 5 for final report)
N_FOLDS = 5
BOOT_N  = 200   # bootstrap iterations for threshold CI

STRATEGIES = ['F1','P80','P85','P90','P95']
PREC_FLOOR = {'P80':0.80, 'P85':0.85, 'P90':0.90, 'P95':0.95}

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

# 55 text features, same as text_cheating_detection.ipynb ALL_FEATURES
ALL_TEXT_FEATURES = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'suspicious_gap_count','suspicious_gap_ratio',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope','energy_mean','energy_std','speaking_rate_std',
    'jitter_local','shimmer_local','hnr_mean',
    'mean_perplexity','burstiness',
]
STYLO_FEATS = ['ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
               'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
               'noun_rate','verb_rate','adj_rate']
TEXT_4GROUP = ALL_TEXT_FEATURES[:40]  # disfluency(6)+stylo(15)+pause(15)+formal_ai(4)? Defined precisely below

TEXT_4GROUP = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
]

print(f'N_SEEDS={N_SEEDS}  N_FOLDS={N_FOLDS}  BOOT_N={BOOT_N}')
print(f'SPW_DEPLOY={SPW_DEPLOY:.2f}  DEPLOY_POS_RATE={DEPLOY_POS_RATE}')
print(f'matplotlib available: {HAS_MPL}')
print(f'Save dir: {SAVE_DIR}')

## 1. Speaker-leakage scan

If the same person appears in both CV and test (even across batches), we're measuring speaker memorisation, not cheating detection. This cell:

1. Looks at raw filenames across `audios2/`, `audios4/`, `audios5/` and prints a sample
2. Tries a few common patterns (name prefix, digit ID, `Q\d+` question code) to extract a speaker-like token
3. Reports overlap across batches

**If the auto-pattern doesn't match your filename convention**, tell me the naming scheme and I'll fix the regex. A no-op here (zero speakers extracted) is a *red flag*, not "all good".

In [ ]:
# --- Step 1: list a sample of filenames per batch ---
print('Sample filenames (first 8 per batch):')
all_files = {}
for b in BATCHES:
    folder = NB_DIR / b
    if not folder.exists():
        print(f'  {b}: folder missing'); continue
    files = sorted(f.name for f in folder.iterdir() if f.is_file())
    all_files[b] = files
    for f in files[:8]:
        print(f'   {b}/  {f}')
    print(f'   ...  total {len(files)} files in {b}/\n')

# --- Step 2: try common speaker-token patterns ---
PATTERNS = [
    (r'^([A-Za-z]+[_ ][A-Za-z]+)',      'name_prefix'),
    (r'^([A-Za-z]+)[_\- ]',              'single_name_prefix'),
    (r'^(\d{4,})',                       'digit_id_prefix'),
    (r'(Q\d{1,3})',                      'question_code'),
    (r'^([A-Za-z0-9]+?)[_\-]Q\d',        'prefix_before_Q'),
]

def extract_tokens(name):
    out = {}
    for pat, tag in PATTERNS:
        m = re.search(pat, name)
        if m: out[tag] = m.group(1)
    return out

records = []
for b, files in all_files.items():
    for fn in files:
        toks = extract_tokens(fn)
        rec = {'batch': b, 'filename': fn}
        rec.update(toks)
        records.append(rec)

rec_df = pd.DataFrame(records)
print('\n--- Pattern hit rates per batch (fraction of files matched by each pattern) ---')
for _, tag in PATTERNS:
    if tag in rec_df.columns:
        for b in BATCHES:
            sub = rec_df[rec_df['batch']==b]
            frac = sub[tag].notna().mean() if len(sub) else 0
            print(f'  {tag:25s}  {b}: {frac*100:5.1f}%  unique={sub[tag].nunique()}')
        print()

# --- Step 3: cross-batch overlap for the best-hit pattern ---
best_pat = None
best_hits = 0
for _, tag in PATTERNS:
    if tag not in rec_df.columns: continue
    h = rec_df[tag].notna().sum()
    if h > best_hits: best_hits, best_pat = h, tag
print(f'Using "{best_pat}" as the best speaker-token proxy.')

if best_pat and best_pat in rec_df.columns:
    valid = rec_df.dropna(subset=[best_pat])
    overlap = (valid.groupby(best_pat)['batch']
                     .apply(lambda s: sorted(set(s)))
                     .apply(lambda s: (len(s), tuple(s))))
    multi = overlap[overlap.apply(lambda t: t[0] > 1)]
    print(f'\nTokens in MORE THAN ONE batch: {len(multi)} / {valid[best_pat].nunique()} unique tokens')
    if len(multi):
        print('  first 20 examples:')
        for tok, (_, bs) in multi.head(20).items():
            print(f'    {tok:30s}  in {bs}')
        print('\n!! LEAKAGE RISK: same token appears across batches. If these are real speakers, '
              'your CV→test numbers are inflated. Tell me the naming scheme so we can build a proper group-aware split.')
    else:
        print('  No cross-batch overlap on this token. Good (or pattern is wrong).')

rec_df.to_csv(SAVE_DIR / 'filename_tokens.csv', index=False)
print(f'\nFull token table saved -> {SAVE_DIR / "filename_tokens.csv"}')

## 2. Load all 3 batches + shared helpers

Reads the existing feature CSVs and returns one DataFrame per batch. Each DataFrame carries:
- `filename`, `label_int`, `batch`
- WavLM pretrained-whole columns (`wavlm_*`)
- Whisper columns (`whisper_*`)
- All 55 text features

Also defines the base-model registry and all evaluation helpers used in every section below.

In [ ]:
# ---------- Data loading ----------
def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def _find_csv(base, candidates):
    for c in candidates:
        p = NB_DIR / c
        if p.exists(): return p
    raise FileNotFoundError(f'None of these exist: {candidates}')

def load_folder(name):
    gt   = load_gt(name)
    text = pd.read_csv(NB_DIR / f'{name}_features.csv')
    wp   = pd.read_csv(_find_csv(name, [f'{name}_whole_pretrained.csv', f'{name}_wavlm_whole.csv']))
    whr  = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df = (gt.merge(text, on='filename', how='inner')
            .merge(wp,   on='filename', how='inner', suffixes=('','_wp'))
            .merge(whr,  on='filename', how='inner', suffixes=('','_wh')))
    df['batch'] = name
    return df

batches = {b: load_folder(b) for b in BATCHES}

# discover column lists from the first batch
first = batches[BATCHES[0]]
WP_COLS    = [c for c in first.columns if c.startswith('wavlm_')
              and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
WH_COLS    = [c for c in first.columns if c.startswith('whisper_')]
TEXT_ALL   = [c for c in ALL_TEXT_FEATURES if c in first.columns]
TEXT_STYLO = [c for c in STYLO_FEATS       if c in first.columns]

for b, df in batches.items():
    y = df['label_int'].values
    print(f'  {b}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}')
print(f'\nWP cols: {len(WP_COLS)}  Whisper cols: {len(WH_COLS)}  Text all: {len(TEXT_ALL)}  Stylo: {len(TEXT_STYLO)}')

# ---------- XGBoost factories ----------
def make_xgb(n_feats, seed=42):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

def fit_score(X_tr, y_tr, X_te, seed=42):
    sc = StandardScaler().fit(X_tr)
    m  = make_xgb(X_tr.shape[1], seed=seed)
    m.fit(sc.transform(X_tr), y_tr)
    return m.predict_proba(sc.transform(X_te))[:,1]

# ---------- Threshold pickers ----------
def best_f1_thr(proba, y, grid=np.arange(0.20, 0.81, 0.01)):
    bt, bf = 0.5, -1.0
    for thr in grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def best_rec_at_prec(proba, y, target, min_tp=3):
    best = None
    for thr in np.arange(0.99, 0.10, -0.01):
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < min_tp: continue
        p = precision_score(y, pred, zero_division=0)
        r = recall_score(y, pred, zero_division=0)
        if p >= target and (best is None or r > best[1]):
            best = (float(thr), float(r), float(p))
    return best if best is not None else (None, None, None)

def pick_thr(proba, y, strategy):
    if strategy == 'F1':
        thr, v = best_f1_thr(proba, y)
        return thr, v
    thr, rec, _ = best_rec_at_prec(proba, y, PREC_FLOOR[strategy])
    return thr, rec

def metrics_at(proba, y, thr):
    if thr is None: return dict(prec=None, rec=None, f1=None)
    pred = (proba >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
    )

# ---------- Base-model registry (feature-subset -> (X-selector, XGB factory)) ----------
def mk_X(cols): return lambda d: d[cols].fillna(0).values

BASE_REGISTRY = {
    'wavlm_wp':       (mk_X(WP_COLS),    lambda s=42: make_xgb(len(WP_COLS), s)),
    'whisper_wp_xgb': (mk_X(WH_COLS),    lambda s=42: make_xgb(len(WH_COLS), s)),
    'text_all':       (mk_X(TEXT_ALL),   lambda s=42: make_xgb(len(TEXT_ALL), s)),
    'text_stylo':     (mk_X(TEXT_STYLO), lambda s=42: make_xgb(len(TEXT_STYLO), s)),
}
print(f'\nBase models in registry: {list(BASE_REGISTRY)}')

## 3. Repeated 5×5-fold CV × 2 rotations, with bootstrap threshold CI

Runs each base model under:
- **Rotation A**: train=[audios2+audios4], CV=audios4, test=audios5
- **Rotation B**: train=[audios2+audios5], CV=audios5, test=audios4

For each rotation, runs `N_SEEDS` × `N_FOLDS` = 15 OOF predictions (at default knobs). All 15 × N_CV predictions per sample are averaged into a single OOF vector per (seed, model) then *concatenated* across seeds for threshold selection.

For each strategy (F1, P80/85/90/95):
- Pick threshold on concatenated OOF
- Bootstrap-resample the (OOF, y) pair `BOOT_N` times → recompute threshold each time → report median + 10/90th percentiles
- Freeze the median threshold, apply to test → report test precision/recall + gap vs CV metric

Output: one table per rotation + a combined "rotation spread" table showing mean and std of test metrics across rotations.

In [ ]:
def repeated_kfold_oof(df_cv, df_always, X_fn, factory, n_seeds=N_SEEDS, n_folds=N_FOLDS):
    """Return OOF vector averaged over n_seeds repeats of n_folds stratified CV.
    On each fold: always-train block is always appended to the fold-train split."""
    y_cv = df_cv['label_int'].values
    oof_sum = np.zeros(len(df_cv)); oof_cnt = np.zeros(len(df_cv))
    for s in range(n_seeds):
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42 + s)
        for tr_idx, va_idx in skf.split(df_cv, y_cv):
            df_tr_fold = df_cv.iloc[tr_idx]
            df_va_fold = df_cv.iloc[va_idx]
            df_tr = pd.concat([df_always, df_tr_fold], ignore_index=True) if df_always is not None and len(df_always) else df_tr_fold
            Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
            Xva = X_fn(df_va_fold)
            sc = StandardScaler().fit(Xtr)
            clf = factory(42 + s)
            clf.fit(sc.transform(Xtr), ytr)
            p = clf.predict_proba(sc.transform(Xva))[:,1]
            oof_sum[va_idx] += p
            oof_cnt[va_idx] += 1
    return oof_sum / np.maximum(oof_cnt, 1)

def bootstrap_thr_ci(proba, y, strategy, n_boot=BOOT_N, seed=0):
    """Bootstrap-resample (proba, y), recompute threshold each time.
    Returns (median, p10, p90, n_valid)."""
    rng = np.random.default_rng(seed)
    n = len(y); idxs = np.arange(n)
    thrs = []
    for _ in range(n_boot):
        b = rng.choice(idxs, size=n, replace=True)
        pb, yb = proba[b], y[b]
        if len(set(yb.tolist())) < 2: continue
        thr, _ = pick_thr(pb, yb, strategy)
        if thr is not None: thrs.append(thr)
    if not thrs:
        return None, None, None, 0
    t = np.array(thrs)
    return float(np.median(t)), float(np.percentile(t,10)), float(np.percentile(t,90)), len(t)

def run_rotation_base(train_folders, cv_target, test_folder, models=None):
    models = models or list(BASE_REGISTRY)
    df_cv  = batches[cv_target].reset_index(drop=True)
    df_al  = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True) \
             if any(b != cv_target for b in train_folders) else None
    df_te  = batches[test_folder].reset_index(drop=True)
    y_cv   = df_cv['label_int'].values
    y_te   = df_te['label_int'].values

    rows = []
    oof_store = {}
    for name in models:
        X_fn, factory = BASE_REGISTRY[name]
        oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory)
        oof_store[name] = oof

        # Refit on full train for test
        df_tr_full = pd.concat([df_al, df_cv], ignore_index=True) if df_al is not None and len(df_al) else df_cv
        Xtr = X_fn(df_tr_full); ytr = df_tr_full['label_int'].values
        Xte = X_fn(df_te)
        sc = StandardScaler().fit(Xtr)
        clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
        p_te = clf.predict_proba(sc.transform(Xte))[:,1]

        for s in STRATEGIES:
            thr_pt, cv_val = pick_thr(oof, y_cv, s)
            thr_md, thr_lo, thr_hi, _ = bootstrap_thr_ci(oof, y_cv, s)
            thr_use = thr_md if thr_md is not None else thr_pt
            te = metrics_at(p_te, y_te, thr_use)
            te_val = te['f1'] if s == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'model': name, 'strategy': s,
                'thr_median': round(thr_use,3) if thr_use is not None else None,
                'thr_p10':    round(thr_lo,3)  if thr_lo  is not None else None,
                'thr_p90':    round(thr_hi,3)  if thr_hi  is not None else None,
                'cv':         round(cv_val,3)  if cv_val  is not None else None,
                'te':         round(te_val,3)  if te_val  is not None else None,
                'te_prec':    round(te['prec'],3) if te['prec'] is not None else None,
                'te_rec':     round(te['rec'],3)  if te['rec']  is not None else None,
                'gap':        round(gap,3)    if gap    is not None else None,
            })
    return pd.DataFrame(rows), oof_store

print('Running Rotation A: train=[a2,a4] CV=a4 test=a5 ...')
df_A, oof_A = run_rotation_base(['audios2','audios4'], 'audios4', 'audios5')
print('Running Rotation B: train=[a2,a5] CV=a5 test=a4 ...')
df_B, oof_B = run_rotation_base(['audios2','audios5'], 'audios5', 'audios4')

print('\n=== ROTATION A (CV=audios4  test=audios5) ===')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(df_A.to_string(index=False, na_rep='  --'))

print('\n=== ROTATION B (CV=audios5  test=audios4) ===')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(df_B.to_string(index=False, na_rep='  --'))

df_A.to_csv(SAVE_DIR / 'rotation_A.csv', index=False)
df_B.to_csv(SAVE_DIR / 'rotation_B.csv', index=False)

In [ ]:
# --- Rotation spread: mean ± std of test metrics across A and B per (model, strategy) ---
merged = (df_A[['model','strategy','cv','te','te_prec','te_rec','gap']].rename(columns=lambda c: c+'_A' if c not in ('model','strategy') else c)
          .merge(df_B[['model','strategy','cv','te','te_prec','te_rec','gap']].rename(columns=lambda c: c+'_B' if c not in ('model','strategy') else c),
                 on=['model','strategy']))

for m in ['cv','te','te_prec','te_rec','gap']:
    a = merged[f'{m}_A']; b = merged[f'{m}_B']
    merged[f'{m}_mean'] = ((a.astype(float) + b.astype(float)) / 2).round(3)
    merged[f'{m}_spread'] = (b.astype(float) - a.astype(float)).abs().round(3)

view_cols = ['model','strategy',
             'cv_A','cv_B','cv_mean',
             'te_A','te_B','te_mean','te_spread',
             'te_prec_A','te_prec_B',
             'gap_A','gap_B','gap_mean']
print('=== ROTATION SPREAD (A vs B) ===')
print('  te_spread = |A - B|.  If > 0.05, the model is batch-variance-dominated;')
print('  any single-rotation number is a coin flip.\n')
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(merged[view_cols].to_string(index=False, na_rep='  --'))

merged.to_csv(SAVE_DIR / 'rotation_spread.csv', index=False)
print(f'\nSaved -> {SAVE_DIR / "rotation_spread.csv"}')

## 4. Per-batch score histograms

For each base model, plot histograms of positive-class probability scores:
- Split by true label (positive vs negative)
- Split by batch (audios2 / audios4 / audios5)

Each model is **refit once on audios2+audios4**, then we score *all three batches* (including the held-out audios5) just to see the full distribution. This is a diagnostic plot, not used for model selection.

If a plot is needed without matplotlib, the cell falls back to printing numeric summaries (median, IQR, top-score negatives per batch).

In [ ]:
# Refit each base model on audios2+audios4, score ALL 3 batches
df_tr = pd.concat([batches['audios2'], batches['audios4']], ignore_index=True)
score_table = {}
for name, (X_fn, factory) in BASE_REGISTRY.items():
    Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
    sc  = StandardScaler().fit(Xtr)
    clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
    per_batch = {}
    for b in BATCHES:
        X = X_fn(batches[b])
        per_batch[b] = clf.predict_proba(sc.transform(X))[:,1]
    score_table[name] = per_batch

# Numeric summary per (model, batch, label)
print('=== Score distribution summary ===\n')
print(f'{"model":<18} {"batch":<10} {"label":<6} {"n":>4} {"median":>8} {"p25":>7} {"p75":>7} {"p95neg":>9}')
for name in score_table:
    for b in BATCHES:
        y = batches[b]['label_int'].values
        p = score_table[name][b]
        for lbl, tag in [(0, 'neg'), (1, 'pos')]:
            mask = (y == lbl)
            if mask.sum() == 0: continue
            vals = p[mask]
            p95_neg = float(np.percentile(p[y==0], 95)) if (y==0).sum() > 0 else np.nan
            print(f'{name:<18} {b:<10} {tag:<6} {int(mask.sum()):>4} '
                  f'{np.median(vals):>8.3f} {np.percentile(vals,25):>7.3f} {np.percentile(vals,75):>7.3f} '
                  f'{p95_neg:>9.3f}')
    print()

# Matplotlib grid (fallback to text if unavailable)
if HAS_MPL:
    nmodels = len(score_table); nbatches = len(BATCHES)
    fig, axes = plt.subplots(nmodels, nbatches, figsize=(4*nbatches, 3*nmodels), sharex=True)
    if nmodels == 1: axes = axes[np.newaxis, :]
    for i, name in enumerate(score_table):
        for j, b in enumerate(BATCHES):
            ax = axes[i, j]
            y = batches[b]['label_int'].values
            p = score_table[name][b]
            if (y==0).sum() > 0: ax.hist(p[y==0], bins=25, alpha=0.5, label='honest', color='steelblue')
            if (y==1).sum() > 0: ax.hist(p[y==1], bins=25, alpha=0.5, label='cheat', color='orangered')
            ax.set_title(f'{name}  /  {b}', fontsize=9)
            ax.set_xlim(0, 1)
            if i == 0 and j == 0: ax.legend(fontsize=8)
    fig.suptitle('Score distributions per (model, batch).  Overlap in the right tail = hard negatives killing rec@P', fontsize=10)
    fig.tight_layout()
    fig.savefig(SAVE_DIR / 'score_histograms.png', dpi=110)
    plt.show()
    print(f'Saved histograms -> {SAVE_DIR / "score_histograms.png"}')
else:
    print('matplotlib unavailable; numeric summary above is the diagnostic.')

## 5. Ensemble-of-seeds on whisper_wp_xgb

XGBoost has stochastic column/row sampling. Averaging probabilities across 5 seeds reduces this variance at no extra data cost.

For Rotation A (train=[a2,a4], CV=a4, test=a5):
- Train whisper_wp_xgb with seeds 42..46 on the full train set, average test probabilities
- Compare F1 / gap to single-seed baseline (already in Section 3)

If the gap shrinks or te F1 rises, ensemble-of-seeds is a free win and should be the default.

In [ ]:
def ensemble_seeds_eval(train_folders, cv_target, test_folder, model_name, seeds=(42,43,44,45,46)):
    X_fn, factory = BASE_REGISTRY[model_name]
    df_cv = batches[cv_target].reset_index(drop=True)
    df_al = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = batches[test_folder].reset_index(drop=True)

    # CV OOF averaged across seeds (already produced by repeated_kfold_oof)
    oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory, n_seeds=len(seeds), n_folds=N_FOLDS)
    y_cv = df_cv['label_int'].values
    y_te = df_te['label_int'].values

    # Single-seed test probabilities
    df_full = pd.concat([df_al, df_cv], ignore_index=True)
    Xtr = X_fn(df_full); ytr = df_full['label_int'].values
    Xte = X_fn(df_te)
    sc = StandardScaler().fit(Xtr)

    single = factory(42); single.fit(sc.transform(Xtr), ytr)
    p_te_single = single.predict_proba(sc.transform(Xte))[:,1]

    p_te_ensemble = np.zeros(len(df_te))
    for s in seeds:
        clf = factory(s); clf.fit(sc.transform(Xtr), ytr)
        p_te_ensemble += clf.predict_proba(sc.transform(Xte))[:,1]
    p_te_ensemble /= len(seeds)

    rows = []
    for tag, p_te in [('single seed=42', p_te_single), (f'ensemble of {len(seeds)} seeds', p_te_ensemble)]:
        for strat in STRATEGIES:
            thr, cv_val = pick_thr(oof, y_cv, strat)
            te = metrics_at(p_te, y_te, thr)
            te_val = te['f1'] if strat == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'variant': tag, 'strategy': strat,
                'thr': round(thr,3) if thr is not None else None,
                'cv':  round(cv_val,3) if cv_val is not None else None,
                'te':  round(te_val,3) if te_val is not None else None,
                'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
                'gap': round(gap,3) if gap is not None else None,
            })
    return pd.DataFrame(rows)

print('Rotation A  — ensemble-of-seeds on whisper_wp_xgb')
df_ens = ensemble_seeds_eval(['audios2','audios4'], 'audios4', 'audios5', 'whisper_wp_xgb')
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(df_ens.to_string(index=False, na_rep='  --'))
df_ens.to_csv(SAVE_DIR / 'ensemble_seeds.csv', index=False)

## 6. PCA diagnostic on Whisper 1024-d features

If Whisper features are noisy/redundant, PCA to fewer dims may *improve* generalisation (smaller gap) while keeping F1. If F1 drops at 80%-variance, information density is actually spread across many dims and full 1024d is needed.

Runs for Rotation A:
- **Baseline**: XGB on full 1024 Whisper dims (= whisper_wp_xgb from registry)
- **PCA@80% / 90% / 95%** variance: PCA fit on the *training* split inside each fold, transform CV and test, train XGB

Reports F1 / P85 / P90 / gap at each dim level. Comparison: does reduction help or hurt?

In [ ]:
def run_pca_rotation(train_folders, cv_target, test_folder, feat_cols, variances=(0.80, 0.90, 0.95)):
    df_cv = batches[cv_target].reset_index(drop=True)
    df_al = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = batches[test_folder].reset_index(drop=True)
    y_cv  = df_cv['label_int'].values
    y_te  = df_te['label_int'].values

    X_al = df_al[feat_cols].fillna(0).values
    X_cv = df_cv[feat_cols].fillna(0).values
    X_te = df_te[feat_cols].fillna(0).values
    y_al = df_al['label_int'].values

    rows = []
    for var_target in variances:
        # Repeated K-fold OOF with fold-scoped PCA
        oof_sum = np.zeros(len(X_cv)); oof_cnt = np.zeros(len(X_cv))
        for s in range(N_SEEDS):
            skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42 + s)
            for tr_i, va_i in skf.split(X_cv, y_cv):
                Xtr = np.vstack([X_al, X_cv[tr_i]]) if len(X_al) else X_cv[tr_i]
                ytr = np.concatenate([y_al, y_cv[tr_i]]) if len(X_al) else y_cv[tr_i]
                sc  = StandardScaler().fit(Xtr)
                pca = PCA(n_components=var_target, random_state=42).fit(sc.transform(Xtr))
                Ztr = pca.transform(sc.transform(Xtr))
                Zva = pca.transform(sc.transform(X_cv[va_i]))
                clf = make_xgb(Ztr.shape[1], seed=42 + s)
                clf.fit(Ztr, ytr)
                oof_sum[va_i] += clf.predict_proba(Zva)[:,1]
                oof_cnt[va_i] += 1
        oof = oof_sum / np.maximum(oof_cnt, 1)

        # Full-train refit → test
        Xtr = np.vstack([X_al, X_cv]) if len(X_al) else X_cv
        ytr = np.concatenate([y_al, y_cv]) if len(X_al) else y_cv
        sc  = StandardScaler().fit(Xtr)
        pca = PCA(n_components=var_target, random_state=42).fit(sc.transform(Xtr))
        Ztr = pca.transform(sc.transform(Xtr))
        Zte = pca.transform(sc.transform(X_te))
        clf = make_xgb(Ztr.shape[1], seed=42); clf.fit(Ztr, ytr)
        p_te = clf.predict_proba(Zte)[:,1]
        n_dims = pca.n_components_

        for strat in STRATEGIES:
            thr, cv_val = pick_thr(oof, y_cv, strat)
            te = metrics_at(p_te, y_te, thr)
            te_val = te['f1'] if strat == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'variant': f'PCA@{int(var_target*100)}% ({n_dims}d)',
                'strategy': strat,
                'cv': round(cv_val,3) if cv_val is not None else None,
                'te': round(te_val,3) if te_val is not None else None,
                'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
                'gap': round(gap,3) if gap is not None else None,
            })

    # Baseline (no PCA) from stored rotation A
    base = df_A[df_A['model']=='whisper_wp_xgb'][['strategy','cv','te','te_prec','gap']].copy()
    base['variant'] = f'no PCA ({len(feat_cols)}d)'
    rows_base = base[['variant','strategy','cv','te','te_prec','gap']].to_dict('records')
    return pd.DataFrame(rows_base + rows)

print('Rotation A — PCA sweep on Whisper features')
df_pca = run_pca_rotation(['audios2','audios4'], 'audios4', 'audios5', WH_COLS)
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(df_pca.to_string(index=False, na_rep='  --'))
df_pca.to_csv(SAVE_DIR / 'pca_diagnostic.csv', index=False)

## 7. WavLM variant comparison (pretrained/finetuned × whole/segmented)

`wavlm_wp` in our base registry is just the pretrained-whole variant. The 4-way notebook has 3 more:
- `whole_finetuned` (768d, fine-tuned WavLM, mean-pool)
- `seg_pretrained` (1536d, pretrained WavLM, segmented mean+std)
- `seg_finetuned` (1536d, fine-tuned, segmented mean+std)

This cell runs repeated-CV on all 4 for Rotation A and picks the best. If one of the fine-tuned/segmented variants has a smaller gap at P85/P90 than `whole_pretrained`, we should swap the default WavLM in the fusion recipe.

**Requires** the `{batch}_whole_pretrained.csv`, `{batch}_whole_finetuned.csv`, `{batch}_seg_pretrained.csv`, `{batch}_seg_finetuned.csv` files to exist (produced by `wavlm_4way_comparison.ipynb`).

In [ ]:
def load_wavlm_variant(batch, csv_suffix):
    p = NB_DIR / f'{batch}_{csv_suffix}.csv'
    if not p.exists():
        return None
    return pd.read_csv(p)

WAVLM_VARIANTS = {
    'whole_pretrained': 'wavlm_',          # 768d
    'whole_finetuned':  'wavlm_',          # 768d
    'seg_pretrained':   'wavlm_mean_|wavlm_std_',  # 1536d (mean+std)
    'seg_finetuned':    'wavlm_mean_|wavlm_std_',
}

def build_variant_df(batch, variant_key):
    feat = load_wavlm_variant(batch, variant_key)
    if feat is None: return None
    gt   = load_gt(batch)
    merged = feat.merge(gt, on='filename', how='inner')
    merged['batch'] = batch
    return merged

def run_wavlm_variant_rotation(variant_key, train_folders, cv_target, test_folder):
    dfs = {b: build_variant_df(b, variant_key) for b in set(train_folders + [test_folder])}
    if any(v is None for v in dfs.values()):
        return None, None
    feat_cols = [c for c in dfs[train_folders[0]].columns if c.startswith('wavlm_')]
    df_cv = dfs[cv_target].reset_index(drop=True)
    df_al = pd.concat([dfs[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = dfs[test_folder].reset_index(drop=True)
    X_fn  = mk_X(feat_cols)
    factory = lambda s=42: make_xgb(len(feat_cols), s)

    oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory)
    y_cv = df_cv['label_int'].values

    df_full = pd.concat([df_al, df_cv], ignore_index=True)
    Xtr = X_fn(df_full); ytr = df_full['label_int'].values
    Xte = X_fn(df_te);   y_te = df_te['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
    p_te = clf.predict_proba(sc.transform(Xte))[:,1]

    rows = []
    for strat in STRATEGIES:
        thr, cv_val = pick_thr(oof, y_cv, strat)
        te = metrics_at(p_te, y_te, thr)
        te_val = te['f1'] if strat == 'F1' else te['rec']
        gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
        rows.append({
            'variant': f'{variant_key} ({len(feat_cols)}d)',
            'strategy': strat,
            'cv': round(cv_val,3) if cv_val is not None else None,
            'te': round(te_val,3) if te_val is not None else None,
            'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
            'gap': round(gap,3) if gap is not None else None,
        })
    return pd.DataFrame(rows), oof

all_variant_rows = []
for vk in WAVLM_VARIANTS:
    print(f'Running variant: {vk} ...')
    dfv, _ = run_wavlm_variant_rotation(vk, ['audios2','audios4'], 'audios4', 'audios5')
    if dfv is None:
        print(f'  skipped ({vk} CSVs not found)')
        continue
    all_variant_rows.append(dfv)

if all_variant_rows:
    df_wvariants = pd.concat(all_variant_rows, ignore_index=True)
    print('\n=== WavLM variant comparison (Rotation A) ===')
    with pd.option_context('display.max_columns', None, 'display.width', 160):
        print(df_wvariants.to_string(index=False, na_rep='  --'))
    df_wvariants.to_csv(SAVE_DIR / 'wavlm_variants.csv', index=False)
else:
    print('No WavLM variant CSVs found. Run wavlm_4way_comparison.ipynb first to produce them.')

## 8. Isotonic calibration + reliability diagram

Fit isotonic on the repeated-CV OOF of whisper_wp_xgb (the strongest base), then apply to test.

**Check 1 — reliability diagram.** Bucket test predictions by calibrated score and compare to actual positive rate in each bucket. A good calibrator has actual ≈ calibrated within ±10 pts.

**Check 2 — recomputed thresholds on calibrated scores.** For each strategy, pick threshold on calibrated CV OOF, apply to calibrated test. Compare gap to the uncalibrated version from Section 3.

A meaningful gap reduction means calibration is worth shipping. A flat result means XGB is already well-calibrated and isotonic adds nothing.

In [ ]:
cal_target = 'whisper_wp_xgb'
oof = oof_A[cal_target]
y_cv_A = batches['audios4']['label_int'].values

# Refit single-seed on full train for test proba
X_fn, factory = BASE_REGISTRY[cal_target]
df_full = pd.concat([batches['audios2'], batches['audios4']], ignore_index=True)
Xtr = X_fn(df_full); ytr = df_full['label_int'].values
Xte = X_fn(batches['audios5']); y_te_A = batches['audios5']['label_int'].values
sc = StandardScaler().fit(Xtr)
clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
p_te = clf.predict_proba(sc.transform(Xte))[:,1]

iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y_cv_A)
p_cv_cal = iso.transform(oof)
p_te_cal = iso.transform(p_te)

# Reliability diagram — test side
bins = np.linspace(0, 1, 11)
bucket_ids = np.digitize(p_te_cal, bins) - 1
bucket_ids = np.clip(bucket_ids, 0, 9)
print('=== Reliability on test (audios5) — whisper_wp_xgb after isotonic ===')
print(f'{"bucket":<15} {"n":>4} {"mean_cal":>9} {"actual_pos_rate":>16}')
for b in range(10):
    m = bucket_ids == b
    if m.sum() == 0: continue
    print(f'[{bins[b]:.1f}-{bins[b+1]:.1f})    {int(m.sum()):>4} {float(p_te_cal[m].mean()):>9.3f} {float(y_te_A[m].mean()):>16.3f}')

# Threshold comparison (raw vs calibrated) per strategy
cmp_rows = []
for strat in STRATEGIES:
    # raw
    thr_r, cv_r = pick_thr(oof, y_cv_A, strat)
    te_r = metrics_at(p_te, y_te_A, thr_r)
    # calibrated
    thr_c, cv_c = pick_thr(p_cv_cal, y_cv_A, strat)
    te_c = metrics_at(p_te_cal, y_te_A, thr_c)
    # metric comparison: F1 or recall
    raw_te  = te_r['f1'] if strat == 'F1' else te_r['rec']
    cal_te  = te_c['f1'] if strat == 'F1' else te_c['rec']
    gap_raw = (cv_r - raw_te) if (cv_r is not None and raw_te is not None) else None
    gap_cal = (cv_c - cal_te) if (cv_c is not None and cal_te is not None) else None
    cmp_rows.append({
        'strategy': strat,
        'raw_thr': round(thr_r,3) if thr_r is not None else None,
        'raw_cv': round(cv_r,3) if cv_r is not None else None,
        'raw_te': round(raw_te,3) if raw_te is not None else None,
        'raw_te_prec': round(te_r['prec'],3) if te_r['prec'] is not None else None,
        'raw_gap': round(gap_raw,3) if gap_raw is not None else None,
        'cal_thr': round(thr_c,3) if thr_c is not None else None,
        'cal_cv': round(cv_c,3) if cv_c is not None else None,
        'cal_te': round(cal_te,3) if cal_te is not None else None,
        'cal_te_prec': round(te_c['prec'],3) if te_c['prec'] is not None else None,
        'cal_gap': round(gap_cal,3) if gap_cal is not None else None,
        'gap_delta': round(abs(gap_cal) - abs(gap_raw), 3) if (gap_raw is not None and gap_cal is not None) else None,
    })
cmp_df = pd.DataFrame(cmp_rows)
print('\n=== Raw vs calibrated thresholds on whisper_wp_xgb (Rotation A) ===')
print('  gap_delta < 0  = calibration reduced |gap|  (good).')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(cmp_df.to_string(index=False, na_rep='  --'))
cmp_df.to_csv(SAVE_DIR / 'calibration_compare.csv', index=False)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(5,4))
    centers = (bins[:-1] + bins[1:]) / 2
    actual = []
    mean_cal = []
    for b in range(10):
        m = bucket_ids == b
        if m.sum() == 0:
            actual.append(np.nan); mean_cal.append(np.nan)
        else:
            actual.append(float(y_te_A[m].mean()))
            mean_cal.append(float(p_te_cal[m].mean()))
    ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='ideal')
    ax.plot(mean_cal, actual, 'o-', color='orangered', label='test')
    ax.set_xlabel('calibrated score (bucket mean)'); ax.set_ylabel('actual positive rate in bucket')
    ax.set_title(f'Reliability diagram — {cal_target} after isotonic (Rotation A)')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(SAVE_DIR / 'reliability.png', dpi=110)
    plt.show()
    print(f'Saved -> {SAVE_DIR / "reliability.png"}')

## 9. Final winner recipe

Prints a compact decision summary:
1. Best *single* base model by average test F1 across both rotations
2. Best base model by P85 test recall (honest high-precision winner)
3. Flag any model where rotation-spread > 0.05 (batch-variance dominated)
4. Short verdict on ensemble, PCA, calibration, WavLM variant — did each help?

In [ ]:
print('='*90)
print('FINAL WINNER RECIPE (from Section 3 rotation spread)')
print('='*90)

# (1) + (2) — best single base by F1 and by P85 recall, averaged across rotations
f1_view = merged[merged['strategy']=='F1'][['model','te_mean','te_spread','gap_mean']]
best_f1 = f1_view.sort_values('te_mean', ascending=False).iloc[0]
print(f'\nBest single model by mean test F1: {best_f1["model"]}')
print(f'   te_F1 mean={best_f1["te_mean"]}  spread={best_f1["te_spread"]}  gap={best_f1["gap_mean"]}')

p85_view = merged[merged['strategy']=='P85'][['model','te_mean','te_spread','gap_mean','te_prec_A','te_prec_B']].copy()
# Only consider models where test precision actually held (>=0.80 on at least one rotation)
p85_view['held_precision'] = ((p85_view['te_prec_A'].astype(float) >= 0.80) |
                               (p85_view['te_prec_B'].astype(float) >= 0.80))
p85_candidates = p85_view[p85_view['held_precision']]
if len(p85_candidates):
    best_p85 = p85_candidates.sort_values('te_mean', ascending=False).iloc[0]
    print(f'\nBest single model by mean test rec@P85 (with precision holding): {best_p85["model"]}')
    print(f'   te_rec mean={best_p85["te_mean"]}  spread={best_p85["te_spread"]}  gap={best_p85["gap_mean"]}')
else:
    print('\nNo single base model held precision ≥ 0.80 on either rotation at P85 strategy.')

# (3) — batch-variance-dominated flag
print('\nModels with te_spread > 0.05 at ANY strategy (batch-variance dominated):')
unstable = merged[merged['te_spread'].astype(float) > 0.05][['model','strategy','te_A','te_B','te_spread']]
if len(unstable):
    with pd.option_context('display.max_columns', None, 'display.width', 140):
        print(unstable.to_string(index=False, na_rep='  --'))
else:
    print('   (none — all models transfer consistently across rotations)')

# (4) — short verdict on each experiment
print('\n' + '-'*90)
print('EXPERIMENT VERDICTS (compare numbers above)')
print('-'*90)
print('  Ensemble-of-seeds (Section 5): read df_ens above — compare single-seed vs 5-seed rows.')
print('     If 5-seed te F1 > single te F1 by >=0.01  -> adopt as default.')
print('  PCA diagnostic (Section 6): read df_pca — compare "no PCA" vs PCA@80/90/95%.')
print('     If PCA@80% matches no-PCA te F1 within 0.01 AND gap is smaller -> adopt PCA@80%.')
print('  WavLM variants (Section 7): if any variant beats whole_pretrained on gap @ P85 by >=0.05,')
print('     swap it in for fusion recipes.')
print('  Calibration (Section 8): if cmp_df shows gap_delta < 0 on >=3 strategies,')
print('     ship the isotonic calibrator alongside the model.')

print('\n' + '='*90)
print('NEXT STEP suggestions (once you send the numbers back)')
print('='*90)
print('  - If Tier-2 wins are small, move to Tier-3 (audio augmentation).')
print('  - If speaker-leakage scan flagged cross-batch tokens, redo all splits group-aware')
print('    and re-run this notebook — expect honest numbers to drop meaningfully.')
print('  - If whisper_wp_xgb is already at ~0.80 F1 with 0.05 gap, Tier-3 may only add a few points.')

print('\nAll tables saved to:', SAVE_DIR)